In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 7

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 1 (tau=59)
[ Info: [sliding] iter 1000/1000000 elapsed=5.6s, rate=0.027, mean=[1.039, 0.00015, 0.356], std=[0.0034, 0.000294, 0.0026] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=10.2s, rate=0.021, mean=[1.027, 0.00013, 0.366], std=[0.0151, 0.000218, 0.0121] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=14.1s, rate=0.025, mean=[0.950, 0.00015, 0.408], std=[0.1107, 0.000189, 0.0583] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=18.0s, rate=0.025, mean=[0.882, 0.00018, 0.446], std=[0.1425, 0.000176, 0.0778] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=21.9s, rate=0.027, mean=[0.842, 0.00020, 0.472], std=[0.1462, 0.000167, 0.0838] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=25.8s, rate=0.028, mean=[0.812, 0.00022, 0.491], std=[0.1461, 0.000161, 0.0849] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=29.8s, rate=0.029, mean=[0.795, 0.00023, 0.506], std=[0.1404, 0.000154, 0.0856] [ADAPT]
[ Info: [sliding] iter 8000/